# MA3001B.502 | Data Wrangling | Eq. Insight

Yuu Ricardo Akachi Tanaka			| A01351969

Viviana Alanis Fraige			    | A01236316

Luis Maximiliano López Ramírez		| A00833321

David Fernando Armendáriz Torres    | A01570813

Kevin Antonio González Díaz 		| A01338316

Cynthia Cristal Quijas Flores       | A01655996

## Environment

In [347]:
import pandas as pd
import numpy as np
pd.DataFrame.iteritems = pd.DataFrame.items
from ydata_profiling import ProfileReport
import seaborn as sns
from scipy.stats import ttest_ind
import scipy.stats as stats
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
from category_encoders import TargetEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
import umap
from ripser import ripser
from persim import PersistenceImager
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings("ignore")

In [348]:
import warnings
warnings.filterwarnings("ignore")

## Data Quality Analysis

### Load & Merge Datasets

In [349]:
try: reservations = pd.read_csv(r'reservaciones_tcabdfront2.csv')
except: reservations = pd.read_csv(r'Datasets\reservaciones_tcabdfront2.csv')

reservations.shape

(203002, 49)

La tabla de datos de reservas contiene 203.000 registros y 49 variables. Esta tabla será la principal del proyecto, ya que contiene todos los registros de reservas de habitaciones de hotel realizadas por los huéspedes.

In [350]:
try:
    agency = pd.read_csv(r'iar_Agencias.csv')
    business = pd.read_csv(r'iar_empresas.csv')
except:
    agency = pd.read_csv(r'Datasets\iar_Agencias.csv')
    business = pd.read_csv(r'Datasets\iar_empresas.csv')

Las tablas iar_Agencias e iar_empresas se unirán a la tabla principal (reservas). Esto se debe a que podrían proporcionar información valiosa sobre las reservas e influir en la probabilidad de cancelación.

Primero, los datos de la reserva se unirán con las variables `Estado_cve` y `Pais_cod` de `iar_Agencia`, utilizando `ID_Agencia` como clave. Esto proporciona más información sobre la agencia utilizada para realizar la reserva, lo que podría influir en las cancelaciones. Se realizó una unión izquierda para conservar toda la información de la reserva.

In [351]:
data = reservations.merge(agency[['ID_Agencia', 'Estado_cve', 'Pais_cod']], on="ID_Agencia", how="left")
data.rename(columns={"Estado_cve": "Estado_cve_agencia", "Pais_cod": "Pais_cod_agencia"}, inplace=True)
data.head()

,ID_Reserva,Fecha_hoy,h_res_fec,h_res_fec_ok,h_res_fec_okt,h_num_per,aa_h_num_per,h_num_adu,aa_h_num_adu,h_num_men,...,h_correo_e,h_nom,h_tfa_total,aa_h_tfa_total,moneda_cve,h_ult_cam_fec,h_ult_cam_fec_ok,h_ult_cam_fec_okt,Estado_cve_agencia,Pais_cod_agencia
0,0,2019-08-16 00:00:00.000,20190816,2019-08-16,2019-08-16 00:00:00.000,2,0,2,0,0,...,NaN,NaN,2659.98,0.0,1,20191110,2019-11-10,2019-11-10 00:00:00.000,EMX,PMX
1,1,2019-10-22 00:00:00.000,20191022,2019-10-22,2019-10-22 00:00:00.000,4,0,4,0,0,...,NaN,NaN,1764.00,0.0,1,20191205,2019-12-05,2019-12-05 00:00:00.000,EMX,PMX
2,2,2019-10-28 00:00:00.000,20191028,2019-10-28,2019-10-28 00:00:00.000,2,0,2,0,0,...,NaN,NaN,2660.04,0.0,1,20191205,2019-12-05,2019-12-05 00:00:00.000,EMX,PMX
3,3,2019-10-28 00:00:00.000,20191028,2019-10-28,2019-10-28 00:00:00.000,4,0,3,0,0,...,NaN,NaN,1995.03,0.0,1,20191211,2019-12-11,2019-12-11 00:00:00.000,EMX,PMX
4,4,2019-10-28 00:00:00.000,20191028,2019-10-28,2019-10-28 00:00:00.000,2,0,2,0,0,...,NaN,NaN,13369.92,0.0,1,20191212,2019-12-12,2019-12-12 00:00:00.000,EQR,PMX


Posteriormente, las variables `Habitaciones_tot` de la tabla `iar_empresas` se unieron a la base de datos de reservas utilizando la variable `ID_empresa` como clave para proporcionar más información sobre el hotel donde se realizó la reserva. También se realizó una unión izquierda para conservar todos los datos de la reserva.

In [352]:
data = data.merge(business[['ID_empresa', 'Habitaciones_tot']], on="ID_empresa", how="left")
data.rename(columns={"Habitaciones_tot": "Habitaciones_tot_hotel"}, inplace=True)
data.head()

,ID_Reserva,Fecha_hoy,h_res_fec,h_res_fec_ok,h_res_fec_okt,h_num_per,aa_h_num_per,h_num_adu,aa_h_num_adu,h_num_men,...,h_nom,h_tfa_total,aa_h_tfa_total,moneda_cve,h_ult_cam_fec,h_ult_cam_fec_ok,h_ult_cam_fec_okt,Estado_cve_agencia,Pais_cod_agencia,Habitaciones_tot_hotel
0,0,2019-08-16 00:00:00.000,20190816,2019-08-16,2019-08-16 00:00:00.000,2,0,2,0,0,...,NaN,2659.98,0.0,1,20191110,2019-11-10,2019-11-10 00:00:00.000,EMX,PMX,735
1,1,2019-10-22 00:00:00.000,20191022,2019-10-22,2019-10-22 00:00:00.000,4,0,4,0,0,...,NaN,1764.00,0.0,1,20191205,2019-12-05,2019-12-05 00:00:00.000,EMX,PMX,735
2,2,2019-10-28 00:00:00.000,20191028,2019-10-28,2019-10-28 00:00:00.000,2,0,2,0,0,...,NaN,2660.04,0.0,1,20191205,2019-12-05,2019-12-05 00:00:00.000,EMX,PMX,735
3,3,2019-10-28 00:00:00.000,20191028,2019-10-28,2019-10-28 00:00:00.000,4,0,3,0,0,...,NaN,1995.03,0.0,1,20191211,2019-12-11,2019-12-11 00:00:00.000,EMX,PMX,735
4,4,2019-10-28 00:00:00.000,20191028,2019-10-28,2019-10-28 00:00:00.000,2,0,2,0,0,...,NaN,13369.92,0.0,1,20191212,2019-12-12,2019-12-12 00:00:00.000,EQR,PMX,735


In [353]:
data.shape

(203002, 52)

Confirmamos que las reservas se mantienen con 203.002 registros y ahora las variables pasaron de 49 a 52 (se agregaron 3 variables más), por lo que las fusiones se realizaron correctamente.

### Referential Integrity Validation

In [354]:
def validate_referential_integrity(foreign_key_df, primary_key_df, foreign_key_column, primary_key_column):
    """
    Valida la integridad referencial entre dos dataframes.

    Parámetros:
    foreign_key_df (pd.DataFrame): DataFrame que contiene la columna de clave externa.
    primary_key_df (pd.DataFrame): DataFrame que contiene la columna de clave primaria.
    foreign_key_column (str): Nombre de la columna de la clave externa en foreign_key_df.
    primary_key_column (str): Nombre de la columna de la clave primaria en primary_key_df.

    Devuelve:
    pd.DataFrame: Filas en foreign_key_df donde la clave externa no coincide con ninguna clave primaria.
    """
    mismatched_rows = foreign_key_df[~foreign_key_df[foreign_key_column].isin(primary_key_df[primary_key_column])]
    return mismatched_rows

# Validar la integridad referencial de 'ID_Agencia' e 'ID_empresa'
agency_mismatches = validate_referential_integrity(reservations, agency, 'ID_Agencia', 'ID_Agencia')
business_mismatches = validate_referential_integrity(reservations, business, 'ID_empresa', 'ID_empresa')

print("Agency mismatches:")
print(agency_mismatches)

print("\nBusiness mismatches:")
print(business_mismatches)

Agency mismatches:
Empty DataFrame
Columns: [ID_Reserva, Fecha_hoy, h_res_fec, h_res_fec_ok, h_res_fec_okt, h_num_per, aa_h_num_per, h_num_adu, aa_h_num_adu, h_num_men, aa_h_num_men, h_num_noc, aa_h_num_noc, h_tot_hab, aa_h_tot_hab, ID_Programa, ID_Paquete, ID_Segmento_Comp, ID_Agencia, ID_empresa, ID_Tipo_Habitacion, ID_canal, h_fec_lld, h_fec_lld_ok, h_fec_lld_okt, h_fec_reg, h_fec_reg_ok, h_fec_reg_okt, h_fec_sda, h_fec_sda_ok, h_fec_sda_okt, ID_Pais_Origen, Cliente_Disp, aa_Cliente_Disp, Reservacion, aa_Reservacion, ID_estatus_reservaciones, h_can_res, h_cod_reserva, h_edo, h_codigop, h_correo_e, h_nom, h_tfa_total, aa_h_tfa_total, moneda_cve, h_ult_cam_fec, h_ult_cam_fec_ok, h_ult_cam_fec_okt]
Index: []

[0 rows x 49 columns]

Business mismatches:
Empty DataFrame
Columns: [ID_Reserva, Fecha_hoy, h_res_fec, h_res_fec_ok, h_res_fec_okt, h_num_per, aa_h_num_per, h_num_adu, aa_h_num_adu, h_num_men, aa_h_num_men, h_num_noc, aa_h_num_noc, h_tot_hab, aa_h_tot_hab, ID_Programa, ID_Paquete

No hay registros donde la clave externa no esté conectada a la clave principal, por lo que la unión se realizó correctamente.

### Variable Identification

In [355]:
data.dtypes

ID_Reserva                    int64
Fecha_hoy                    object
h_res_fec                     int64
h_res_fec_ok                 object
h_res_fec_okt                object
h_num_per                     int64
aa_h_num_per                  int64
h_num_adu                     int64
aa_h_num_adu                  int64
h_num_men                     int64
aa_h_num_men                  int64
h_num_noc                     int64
aa_h_num_noc                  int64
h_tot_hab                     int64
aa_h_tot_hab                  int64
ID_Programa                   int64
ID_Paquete                    int64
ID_Segmento_Comp              int64
ID_Agencia                    int64
ID_empresa                    int64
ID_Tipo_Habitacion            int64
ID_canal                      int64
h_fec_lld                    object
h_fec_lld_ok                 object
h_fec_lld_okt                object
h_fec_reg                    object
h_fec_reg_ok                 object
h_fec_reg_okt               

Observamos que algunos tipos de datos son incorrectos. Las variables deberían ser las siguientes:
* h_num_per: Int64
* aa_h_num_per: Int64
* h_num_adu: Int64
* aa_h_num_adu: Int64
* h_num_men: Int64
* aa_h_num_men: Int64
* h_num_noc: Int64
* aa_h_num_noc: Int64
* h_tot_hab: Int64
* aa_h_tot_hab: Int64
* Habitaciones_tot_hotel: Int64
* h_tfa_total: float64
* aa_h_tfa_total: float64
* ID_Reserva: category
* ID_Programa: category
* ID_Paquete: category
* ID_Segmento_Comp: categoría
* ID_Agencia: categoría
* ID_empresa: categoría
* ID_Tipo_Habitación: categoría
* ID_canal: categoría
* ID_País_Origen: categoría
* Cliente_Disp: categoría
* aa_Cliente_Disp: categoría
* ID_estado_reservas: categoría
* h_can_res: categoría
* h_cod_reserva: categoría
* h_edo: categoría
* h_codigop: categoría
* moneda_cve: categoría
* Estado_cve_agencia: categoría
* País_cod_agencia: categoría
* Reserva: bool
* aa_Reserva: bool
* h_correo_e: str
* h_nom: str
* h_fec_lld: datetime
* h_fec_lld: datetime
* h_fec_lld_ok: datetime
* h_fec_lld_okt: fecha y hora
* h_fec_reg: fecha y hora
* h_fec_reg_ok: fecha y hora
* h_fec_reg_okt: fecha y hora
* h_fec_sda: fecha y hora
* h_fec_sda_ok: fecha y hora

### Data Correction Type

In [356]:
int_cols = ['h_num_per', 'aa_h_num_per', 'h_num_adu', 'aa_h_num_adu', 'h_num_men', 
        'aa_h_num_men', 'h_num_noc', 'aa_h_num_noc', 'h_tot_hab', 'aa_h_tot_hab', 'Habitaciones_tot_hotel']
data[int_cols] = data[int_cols].astype('Int64')

float_cols = ['h_tfa_total', 'aa_h_tfa_total']
data[float_cols] = data[float_cols].astype('float64')

category_cols = ['ID_Reserva', 'ID_Programa', 'ID_Paquete', 'ID_Segmento_Comp', 
                  'ID_Agencia', 'ID_empresa', 'ID_Tipo_Habitacion', 'ID_canal', 
                  'ID_Pais_Origen', 'Cliente_Disp', 'aa_Cliente_Disp', 
                  'ID_estatus_reservaciones', 'h_can_res', 'h_cod_reserva', 
                  'h_edo', 'h_codigop', 'moneda_cve', 'Estado_cve_agencia', 'Pais_cod_agencia']
data[category_cols] = data[category_cols].astype('category')

binary_cols = ['Reservacion', 'aa_Reservacion']
data[binary_cols] = data[binary_cols].astype('bool')

string_cols = ['h_correo_e', 'h_nom']
data[string_cols] = data[string_cols].astype('str')

In [357]:
data = data.applymap(lambda x: x.strip() if isinstance(x, str) else x)

date_columns = ["h_res_fec", "h_fec_lld", "h_fec_reg", "h_fec_sda", "h_ult_cam_fec","h_res_fec_ok", "h_fec_lld_ok", "h_fec_reg_ok", "h_ult_cam_fec_ok", 
                "h_fec_sda_ok", "Fecha_hoy", "h_res_fec_okt", "h_fec_lld_okt", "h_fec_reg_okt", "h_fec_sda_okt", "h_ult_cam_fec_okt"]
data[date_columns] = data[date_columns].apply(lambda x: pd.to_datetime(x, format="%Y-%m-%d %H:%M:%S.%f", errors="ignore"))
data[date_columns] = data[date_columns].apply(lambda x: pd.to_datetime(x, format="%Y%m%d", errors="ignore"))
data[date_columns] = data[date_columns].apply(lambda x: pd.to_datetime(x, format="%Y-%m-%d", errors="ignore"))

In [358]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203002 entries, 0 to 203001
Data columns (total 52 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   ID_Reserva                203002 non-null  category      
 1   Fecha_hoy                 203002 non-null  datetime64[ns]
 2   h_res_fec                 203002 non-null  datetime64[ns]
 3   h_res_fec_ok              203002 non-null  datetime64[ns]
 4   h_res_fec_okt             203002 non-null  datetime64[ns]
 5   h_num_per                 203002 non-null  int64         
 6   aa_h_num_per              203002 non-null  int64         
 7   h_num_adu                 203002 non-null  int64         
 8   aa_h_num_adu              203002 non-null  int64         
 9   h_num_men                 203002 non-null  int64         
 10  aa_h_num_men              203002 non-null  int64         
 11  h_num_noc                 203002 non-null  int64         
 12  aa

### Univariate Analysis

In [359]:
data_1 = data.copy()

La distribución de las variables numéricas se comprobó mediante histogramas.

In [ ]:
plt.style.use('seaborn-v0_8-colorblind')
data_1[int_cols + float_cols].hist(figsize=(15,15),histtype='stepfilled',layout=(7,2),bins=10)

In [361]:
data_1['h_tfa_total'].min(), data_1['aa_h_tfa_total'].min()

(-30910.0, -30910.0)

Todo parece correcto; todas las variables son positivas excepto `h_tfa_total` y `aa_h_tfa_total`, ya que son valores monetarios y podrían representar pérdidas. Ninguna variable se comporta con normalidad; esto podría deberse a que hay muchos valores 0.

### Identify Outliers

In [ ]:
plt.style.use('seaborn-v0_8-colorblind')
fig, axes = plt.subplots(nrows=7, ncols=2, figsize=(15, 15))
axes = axes.flatten()
for i, col in enumerate(int_cols + float_cols):
    sns.boxplot(y=data_1[col], ax=axes[i])
    axes[i].set_title(col)

plt.tight_layout()
plt.show()

In [363]:
data_1[int_cols + float_cols].describe()

,h_num_per,aa_h_num_per,h_num_adu,aa_h_num_adu,h_num_men,aa_h_num_men,h_num_noc,aa_h_num_noc,h_tot_hab,aa_h_tot_hab,Habitaciones_tot_hotel,h_tfa_total,aa_h_tfa_total
count,203002.000000,203002.000000,203002.000000,203002.000000,203002.000000,203002.000000,203002.000000,203002.000000,203002.000000,203002.000000,203002.0,2.030020e+05,2.030020e+05
mean,1.185865,1.170550,1.094708,1.079694,0.034108,0.033734,1.680028,1.656043,0.505581,0.498586,735.0,4.106728e+03,4.039369e+03
std,1.369629,1.368073,1.207076,1.204694,0.265187,0.264043,2.866189,2.847958,0.508751,0.508489,0.0,6.501707e+03,6.443027e+03
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,735.0,-3.091000e+04,-3.091000e+04
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,735.0,0.000000e+00,0.000000e+00
50%,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,735.0,0.000000e+00,0.000000e+00
75%,2.000000,2.000000,2.000000,2.000000,0.000000,0.000000,3.000000,3.000000,1.000000,1.000000,735.0,7.592000e+03,7.494000e+03
max,32.000000,32.000000,15.000000,15.000000,15.000000,15.000000,687.000000,687.000000,6.000000,6.000000,735.0,1.033056e+06,1.033056e+06


Al realizar los diagramas de caja y bigotes, observamos varios valores atípicos. Sin embargo, estos valores podrían ser realistas y no erróneos, por lo que podrían brindarnos información valiosa para predecir las probabilidades de cancelación. Por ejemplo, si la variable `h_num_per` tiene un máximo de 32, esto podría indicar que se realizó una reserva muy grande debido a un evento, incluso reservando varias habitaciones. Esto se observa en las variables `h_tot_hab` y `aa_h_tot_hab`, que tienen un máximo de 6, al igual que `aa_h_num_per`, `h_num_adu`, `aa_h_num_adu`, `h_num_men` y `aa_h_num_men`. Incluso podría haber habitaciones tipo cabaña. Para las variables `h_num_noc` y `aa_h_num_noc`, observamos que tienen un máximo de 687, lo que podría representar una estancia de 687 noches, lo cual podría ser inusual. Sin embargo, hay personas que viven en hoteles. Por lo tanto, por el momento, se decidió dejar los valores atípicos.

### Continous Variable Analysis

In [ ]:
# Crear gráficos de densidad uno al lado del otro
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Primer gráfico de densidad para h_tfa_total
sns.kdeplot(data_1['h_tfa_total'], fill=True, bw_method=1, ax=axes[0], label='h_tfa_total')
axes[0].set_title('Density Plot for h_tfa_total')
axes[0].legend()

# Segundo gráfico de densidad para aa_tfa_total
sns.kdeplot(data_1['aa_h_tfa_total'], fill=True, bw_method=1, ax=axes[1], label='aa_h_tfa_total', color='orange')
axes[1].set_title('Density Plot for aa_h_tfa_total')
axes[1].legend()

plt.tight_layout()
plt.show()

Como puede ver, no parecen tener una distribución normal. A continuación, se eliminan los valores atípicos para comprobar si hay alguna diferencia.

In [365]:
# Descartar valores atípicos según la regla 1,5*RIC
Q1_h_tfa_total = data_1['h_tfa_total'].quantile(0.25)
Q3_h_tfa_total = data_1['h_tfa_total'].quantile(0.75)
IQR_h_tfa_total = Q3_h_tfa_total - Q1_h_tfa_total

Q1_aa_h_tfa_total = data_1['aa_h_tfa_total'].quantile(0.25)
Q3_aa_h_tfa_total = data_1['aa_h_tfa_total'].quantile(0.75)
IQR_aa_h_tfa_total = Q3_aa_h_tfa_total - Q1_aa_h_tfa_total

# Filtrar los datos para eliminar valores atípicos
filtered_data = data_1[
    (data_1['h_tfa_total'] >= Q1_h_tfa_total - 1.5 * IQR_h_tfa_total) &
    (data_1['h_tfa_total'] <= Q3_h_tfa_total + 1.5 * IQR_h_tfa_total) &
    (data_1['aa_h_tfa_total'] >= Q1_aa_h_tfa_total - 1.5 * IQR_aa_h_tfa_total) &
    (data_1['aa_h_tfa_total'] <= Q3_aa_h_tfa_total + 1.5 * IQR_aa_h_tfa_total)
]

In [ ]:
# Crear gráficos de densidad uno al lado del otro
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Primer gráfico de densidad para h_tfa_total
sns.kdeplot(filtered_data['h_tfa_total'], fill=True, bw_method=1, ax=axes[0], label='h_tfa_total')
axes[0].set_title('Density Plot for h_tfa_total')
axes[0].legend()

# Segundo gráfico de densidad para aa_tfa_total
sns.kdeplot(filtered_data['aa_h_tfa_total'], fill=True, bw_method=1, ax=axes[1], label='aa_h_tfa_total', color='orange')
axes[1].set_title('Density Plot for aa_h_tfa_total')
axes[1].legend()

plt.tight_layout()
plt.show()

In [367]:
# Comprobar si la distribución es normal utilizando la prueba de Shapiro-Wilk
from scipy.stats import shapiro
stat_h_tfa_total, p_h_tfa_total = shapiro(filtered_data['h_tfa_total'])
stat_aa_h_tfa_total, p_aa_h_tfa_total = shapiro(filtered_data['aa_h_tfa_total'])

print("\nShapiro-Wilk Normality Test for h_tfa_total:")
print(f"Statistic: {stat_h_tfa_total}, p-value: {p_h_tfa_total}")
print("Normal distribution" if p_h_tfa_total > 0.05 else "Not a normal distribution")

print("\nShapiro-Wilk Normality Test for aa_h_tfa_total:")
print(f"Statistic: {stat_aa_h_tfa_total}, p-value: {p_aa_h_tfa_total}")
print("Normal distribution" if p_aa_h_tfa_total > 0.05 else "Not a normal distribution")


Shapiro-Wilk Normality Test for h_tfa_total:
Statistic: 0.7772333025932312, p-value: 0.0
Not a normal distribution

Shapiro-Wilk Normality Test for aa_h_tfa_total:
Statistic: 0.7717701196670532, p-value: 0.0
Not a normal distribution


Al observar los gráficos de densidad, parecen tener una curva en forma de campana; sin embargo, se observa claramente una ligera asimetría. Tras realizar la prueba de normalidad de Shapiro-Wilk, se observa que existe suficiente evidencia para rechazar $H_0$, por lo que estas variables continuas no se distribuyen normalmente.

### Pandas Data Profiling

In [ ]:
profile_report = ProfileReport(data_1, title="Profiling Report")
profile_report.to_file("Profile_report.html")

Lo principal que observamos con esta herramienta es que hay muchos ceros en muchas variables. Necesitaremos investigar el significado del 0 para realizar un análisis correcto y eficaz que nos ayude más adelante en el modelo.

## Data Cleansing

In [369]:
data_1 = data.copy()

### Indentificar Variables cardinalidad 0 o 1

In [370]:
unique_counts = data_1.nunique().sort_values()

for column, count in unique_counts.items():
    print(f"Column '{column}' has {count} unique values")

Column 'Habitaciones_tot_hotel' has 1 unique values
Column 'h_correo_e' has 1 unique values
Column 'h_nom' has 1 unique values
Column 'moneda_cve' has 1 unique values
Column 'ID_empresa' has 1 unique values
Column 'aa_Reservacion' has 2 unique values
Column 'ID_Programa' has 2 unique values
Column 'Reservacion' has 2 unique values
Column 'ID_Pais_Origen' has 4 unique values
Column 'Pais_cod_agencia' has 5 unique values
Column 'ID_Paquete' has 6 unique values
Column 'h_tot_hab' has 7 unique values
Column 'aa_h_tot_hab' has 7 unique values
Column 'ID_estatus_reservaciones' has 8 unique values
Column 'h_num_men' has 10 unique values
Column 'aa_h_num_men' has 10 unique values
Column 'ID_canal' has 14 unique values
Column 'h_can_res' has 15 unique values
Column 'Cliente_Disp' has 16 unique values
Column 'h_num_adu' has 16 unique values
Column 'aa_h_num_adu' has 16 unique values
Column 'aa_Cliente_Disp' has 16 unique values
Column 'ID_Segmento_Comp' has 20 unique values
Column 'h_num_per' ha

Las siguientes variables no proporcionan información ya que sólo tienen un valor posible: |`h_correo_e`, `h_nom`, `moneda_cve`, `ID_empresa`.

In [371]:
data_1 = data_1.drop(['h_correo_e', 'h_nom', 'moneda_cve', 'ID_empresa', 'Habitaciones_tot_hotel'], axis=1)
data_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203002 entries, 0 to 203001
Data columns (total 47 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   ID_Reserva                203002 non-null  category      
 1   Fecha_hoy                 203002 non-null  datetime64[ns]
 2   h_res_fec                 203002 non-null  datetime64[ns]
 3   h_res_fec_ok              203002 non-null  datetime64[ns]
 4   h_res_fec_okt             203002 non-null  datetime64[ns]
 5   h_num_per                 203002 non-null  int64         
 6   aa_h_num_per              203002 non-null  int64         
 7   h_num_adu                 203002 non-null  int64         
 8   aa_h_num_adu              203002 non-null  int64         
 9   h_num_men                 203002 non-null  int64         
 10  aa_h_num_men              203002 non-null  int64         
 11  h_num_noc                 203002 non-null  int64         
 12  aa

### Null Value Management

En primera, los datos que eran solo un espacio vacío se reemplazan con valores nan.

In [372]:
data_1 = data_1.replace('', np.nan)
data_1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203002 entries, 0 to 203001
Data columns (total 47 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   ID_Reserva                203002 non-null  category      
 1   Fecha_hoy                 203002 non-null  datetime64[ns]
 2   h_res_fec                 203002 non-null  datetime64[ns]
 3   h_res_fec_ok              203002 non-null  datetime64[ns]
 4   h_res_fec_okt             203002 non-null  datetime64[ns]
 5   h_num_per                 203002 non-null  int64         
 6   aa_h_num_per              203002 non-null  int64         
 7   h_num_adu                 203002 non-null  int64         
 8   aa_h_num_adu              203002 non-null  int64         
 9   h_num_men                 203002 non-null  int64         
 10  aa_h_num_men              203002 non-null  int64         
 11  h_num_noc                 203002 non-null  int64         
 12  aa

Debido a su elevado número de valores faltantes, se eliminarán estas columnas: `h_cod_reserva`, `h_codigop`.

In [373]:
data_1 = data_1.drop(['h_cod_reserva', 'h_codigop'], axis=1)

Después de eliminar las columnas anteriores, se eliminaron las entradas con valores nulos ya que no representan un alto porcentaje de los datos.

In [374]:
data_1 = data_1.dropna()
data_1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 201358 entries, 0 to 203001
Data columns (total 45 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   ID_Reserva                201358 non-null  category      
 1   Fecha_hoy                 201358 non-null  datetime64[ns]
 2   h_res_fec                 201358 non-null  datetime64[ns]
 3   h_res_fec_ok              201358 non-null  datetime64[ns]
 4   h_res_fec_okt             201358 non-null  datetime64[ns]
 5   h_num_per                 201358 non-null  int64         
 6   aa_h_num_per              201358 non-null  int64         
 7   h_num_adu                 201358 non-null  int64         
 8   aa_h_num_adu              201358 non-null  int64         
 9   h_num_men                 201358 non-null  int64         
 10  aa_h_num_men              201358 non-null  int64         
 11  h_num_noc                 201358 non-null  int64         
 12  aa_h_nu

In [375]:
percentage = (203002-201358)/203002 * 100
print(f"Percentage of dropped data: {percentage:.2f}%")

Percentage of dropped data: 0.81%


### Identify Redundant (Identical) Columns

In [376]:
data_1['Reservacion'].value_counts()

Reservacion
True     101366
False     99992
Name: count, dtype: int64

In [377]:
data_1[data_1['Reservacion']==1]['aa_Reservacion'].value_counts()

aa_Reservacion
False    101366
Name: count, dtype: int64

In [378]:
data_1[data_1['aa_Reservacion']==1]['Reservacion'].value_counts()

Reservacion
False    99992
Name: count, dtype: int64

Las variables `Reserva` y `aa_Reserva` contienen la misma información, reflejada. aa_Reserva será descartada.

In [379]:
data_1 = data_1.drop(['aa_Reservacion'], axis=1)

Comprobamos si algunas columnas eran exactamente iguales. De ser así, se debería eliminar una, ya que tener columnas idénticas no sirve de nada.

In [380]:
identical_columns = []
for col1 in data_1.columns:
    for col2 in data_1.columns:
        if col1 != col2 and data_1[col1].equals(data_1[col2]):
            identical_columns.append((col1, col2))

print("Identical columns:")
for col1, col2 in identical_columns:
    print(f"'{col1}' is identica to '{col2}'")

Identical columns:
'h_res_fec' is identica to 'h_res_fec_ok'
'h_res_fec' is identica to 'h_res_fec_okt'
'h_res_fec_ok' is identica to 'h_res_fec'
'h_res_fec_ok' is identica to 'h_res_fec_okt'
'h_res_fec_okt' is identica to 'h_res_fec'
'h_res_fec_okt' is identica to 'h_res_fec_ok'
'h_fec_lld' is identica to 'h_fec_lld_ok'
'h_fec_lld' is identica to 'h_fec_lld_okt'
'h_fec_lld_ok' is identica to 'h_fec_lld'
'h_fec_lld_ok' is identica to 'h_fec_lld_okt'
'h_fec_lld_okt' is identica to 'h_fec_lld'
'h_fec_lld_okt' is identica to 'h_fec_lld_ok'
'h_fec_reg' is identica to 'h_fec_reg_ok'
'h_fec_reg' is identica to 'h_fec_reg_okt'
'h_fec_reg_ok' is identica to 'h_fec_reg'
'h_fec_reg_ok' is identica to 'h_fec_reg_okt'
'h_fec_reg_okt' is identica to 'h_fec_reg'
'h_fec_reg_okt' is identica to 'h_fec_reg_ok'
'h_fec_sda' is identica to 'h_fec_sda_ok'
'h_fec_sda' is identica to 'h_fec_sda_okt'
'h_fec_sda_ok' is identica to 'h_fec_sda'
'h_fec_sda_ok' is identica to 'h_fec_sda_okt'
'h_fec_sda_okt' is ide

Se eliminarán las siguientes columnas:

Mantener        | Eliminar:
- h_res_fec     | h_res_fec_ok, h_res_fec_okt
- h_num_adu     | Cliente_Disp
- aa_h_num_adu  | aa_Cliente_Disp
- h_fec_lld     | h_fec_lld_ok, h_fec_lld_okt
- h_fec_reg     | h_fec_reg_ok, h_fec_reg_okt
- h_fec_sda     | h_fec_sda_ok, h_fec_sda_okt
- h_ult_cam_fec | h_ult_cam_fec_ok, h_ult_cam_fec_okt

In [381]:
data_1 = data_1.drop(['h_res_fec_ok', 'h_res_fec_okt', 'Cliente_Disp', 'aa_Cliente_Disp', 'h_fec_lld_ok', 'h_fec_lld_okt', 'h_fec_reg_ok',
                  'h_fec_reg_okt', 'h_fec_sda_ok', 'h_fec_sda_okt', 'h_ult_cam_fec_ok', 'h_ult_cam_fec_okt'], axis = 1)

In [382]:
data_1.shape

(201358, 32)

In [383]:
data_1['ID_Programa'].value_counts()

ID_Programa
1    201358
0         0
Name: count, dtype: int64

Después de eliminar entradas con valores nulos, `ID_Programa` se convirtió en una constante y, por lo tanto, no proporciona información. 

In [384]:
data_1 = data_1.drop('ID_Programa', axis=1)

### Duplicate Entries

In [385]:
duplicate_rows = data_1[data_1.duplicated()]
print("Duplicate Rows:")
print(duplicate_rows)

Duplicate Rows:
Empty DataFrame
Columns: [ID_Reserva, Fecha_hoy, h_res_fec, h_num_per, aa_h_num_per, h_num_adu, aa_h_num_adu, h_num_men, aa_h_num_men, h_num_noc, aa_h_num_noc, h_tot_hab, aa_h_tot_hab, ID_Paquete, ID_Segmento_Comp, ID_Agencia, ID_Tipo_Habitacion, ID_canal, h_fec_lld, h_fec_reg, h_fec_sda, ID_Pais_Origen, Reservacion, ID_estatus_reservaciones, h_can_res, h_edo, h_tfa_total, aa_h_tfa_total, h_ult_cam_fec, Estado_cve_agencia, Pais_cod_agencia]
Index: []

[0 rows x 31 columns]


No hay entradas duplicadas.

## Feature Engineering

### Analyze Target Variable

In [386]:
data_2 = data_1.copy()

In [387]:
target = data_2['ID_estatus_reservaciones'].copy()
target = target.apply(lambda x: 1 if x in [2, 3] else 0)
percentages = target.value_counts(normalize=True) * 100
print(percentages)

ID_estatus_reservaciones
0    83.070948
1    16.929052
Name: proportion, dtype: float64


Si consideramos las Cancelaciones y las No Presentaciones como cancelaciones, y los demás estados como no cancelados, el 83 % de las inscripciones no se cancelaron y el 16 % sí.

Ahora solo consideraremos los registros con la variable `ID_estado_reservas` con 2 (Reserva Cancelada), 3 (No Presentarse), 9 (En el Hotel) y 9 (Salida). Esto se debe a que "Salida" indica que la visita ya se realizó; los demás campos aún no pueden determinar si se cancelará o no.

In [388]:
data_3 = data_2[data_2['ID_estatus_reservaciones'].isin([2, 3, 9, 8])].copy()

In [389]:
target = data_3['ID_estatus_reservaciones'].copy()
target = target.apply(lambda x: 1 if x in [2, 3] else 0)
percentages = target.value_counts(normalize=True) * 100
print(percentages)

ID_estatus_reservaciones
0    79.372235
1    20.627765
Name: proportion, dtype: float64


Haciendo esto, tenemos 79% de datos como no cancelados y el 21% como cancelados, aumentamos el porcentaje de datos cancelados.

### Initial Data Selection

Se eliminarán todas las variables que hagan referencia a años anteriores (aa).

In [390]:
aa_cols = [col for col in data_3.columns if col.startswith('aa')]
aa_cols

['aa_h_num_per',
 'aa_h_num_adu',
 'aa_h_num_men',
 'aa_h_num_noc',
 'aa_h_tot_hab',
 'aa_h_tfa_total']

In [391]:
data_3 = data_3.drop(aa_cols, axis=1)

Después de discutirlo con TCA Software Solutions, las siguientes variables se descartarán por falta de relevancia en el proyecto: `Reserva`, `h_num_noc`, `h_fec_sda`, `Fecha_hoy`, `h_ult_cam_fec`

In [392]:
data_3 = data_3.drop(['Reservacion', 'h_fec_sda', 'Fecha_hoy', 'h_ult_cam_fec', 'h_fec_reg'], axis=1)

### Feature Engineering

#### Calcular tarifa por noche por habitación.4

In [393]:
tfa_pHab = data_3.groupby(['h_num_noc', 'ID_Tipo_Habitacion'])['h_tfa_total'].sum().reset_index()

tfa_pHab = tfa_pHab[tfa_pHab['h_num_noc'] != 0]

In [394]:
tfa_pHab['avg_noc'] = tfa_pHab['h_tfa_total'] / tfa_pHab['h_num_noc']

tfa_pHab = tfa_pHab[tfa_pHab['avg_noc'] != 0]

In [395]:
import json
avg_noc_dict = tfa_pHab.groupby('ID_Tipo_Habitacion')['avg_noc'].mean().to_dict()
with open("avg_noc_dict.json", "w") as f:
    json.dump(avg_noc_dict, f)


In [396]:
data_3['tarif'] = data_3['ID_Tipo_Habitacion'].map(avg_noc_dict)

#### Calcular con qué antelación se hizo la reserva

In [397]:
data_3['days_diff_res_lld'] = (data_3['h_fec_lld'] - data_3['h_res_fec']).dt.days

data_3['days_diff_res_lld'].value_counts()

days_diff_res_lld
 1      7842
 0      7703
 2      5128
 3      3748
 4      3632
        ... 
 359       2
 402       2
 529       2
 356       2
-6         2
Name: count, Length: 357, dtype: int64

In [398]:
data_3 = data_3[data_3['days_diff_res_lld'] >= 0]
data_3['days_diff_res_lld'].value_counts()

days_diff_res_lld
1      7842
0      7703
2      5128
3      3748
4      3632
       ... 
338       2
361       2
356       2
345       2
402       2
Name: count, Length: 355, dtype: int64

#### Reescribir `h_num_per` como la suma de `h_num_adu` y `h_num_men`

In [399]:
data_3 = data_3.drop(['h_num_per'], axis=1)

data_3['h_num_per'] = data_3['h_num_adu'] + data_3['h_num_men']

##### Reescrbir ID_Pais_Origen como nacional(1) o extranjero(0).

In [400]:
data_3['Pais_Origen'] = data_3['ID_Pais_Origen'].apply(lambda x: 1 if x == 157 else 0)
data_3.drop(['ID_Pais_Origen'], axis=1, inplace=True)
data_3['Pais_Origen'].value_counts()

Pais_Origen
1    163735
0      1464
Name: count, dtype: int64

##### Reescribe la variable de destino como cancelación (2,3 -> 1) y no cancelación (8,9 -> 0)

In [401]:
data_3['ID_estatus_reservaciones'] = data_3['ID_estatus_reservaciones'].replace({2: 1, 3: 1, 9: 0, 8: 0})

data_3

,ID_Reserva,h_res_fec,h_num_adu,h_num_men,h_num_noc,h_tot_hab,ID_Paquete,ID_Segmento_Comp,ID_Agencia,ID_Tipo_Habitacion,...,ID_estatus_reservaciones,h_can_res,h_edo,h_tfa_total,Estado_cve_agencia,Pais_cod_agencia,tarif,days_diff_res_lld,h_num_per,Pais_Origen
0,0,2019-08-16,2,0,3,1,1,14,112,25,...,0,09,EMX,2659.98,EMX,PMX,2.693467e+05,83,2,1
1,1,2019-10-22,4,0,4,1,2,14,112,25,...,0,09,EGT,1764.00,EMX,PMX,2.693467e+05,40,4,1
2,2,2019-10-28,2,0,4,1,1,14,112,23,...,0,09,EMC,2660.04,EMX,PMX,1.988246e+05,34,2,1
3,3,2019-10-28,3,0,3,1,2,14,112,24,...,0,09,EGT,1995.03,EMX,PMX,1.742958e+05,41,3,1
4,4,2019-10-28,2,0,4,1,1,5,14,4,...,0,12,EMX,13369.92,EQR,PMX,3.608777e+06,41,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202997,202997,2019-04-10,0,0,0,0,2,14,112,19,...,1,00,EMC,0.00,EMX,PMX,2.748716e+06,0,0,0
202998,202998,2019-06-17,0,0,0,0,1,14,112,13,...,0,00,EGR,0.00,EMX,PMX,9.116025e+05,0,0,0
202999,202999,2019-05-30,0,0,0,0,2,14,112,17,...,0,00,EMC,0.00,EMX,PMX,2.525819e+06,0,0,0
203000,203000,2019-08-10,0,0,0,0,1,5,14,6,...,0,00,EMC,0.00,EQR,PMX,9.400164e+05,0,0,0


In [402]:
data_3['Cancellation'] = data_3['ID_estatus_reservaciones'].apply(lambda x: 1 if x == 1 else 0)

data_3['Cancellation'].value_counts()

Cancellation
0    131113
1     34086
Name: count, dtype: int64

### Elimine las variables que quedaron obsoletas después de la ingeniería de características

In [403]:
data_3 = data_3.drop(["ID_estatus_reservaciones"], axis=1)
data_3 = data_3.drop(["h_tfa_total"], axis=1)
data_3 = data_3.drop(["h_num_noc"], axis=1)
data_3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 165199 entries, 0 to 203001
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   ID_Reserva          165199 non-null  category      
 1   h_res_fec           165199 non-null  datetime64[ns]
 2   h_num_adu           165199 non-null  int64         
 3   h_num_men           165199 non-null  int64         
 4   h_tot_hab           165199 non-null  int64         
 5   ID_Paquete          165199 non-null  category      
 6   ID_Segmento_Comp    165199 non-null  category      
 7   ID_Agencia          165199 non-null  category      
 8   ID_Tipo_Habitacion  165199 non-null  category      
 9   ID_canal            165199 non-null  category      
 10  h_fec_lld           165199 non-null  datetime64[ns]
 11  h_can_res           165199 non-null  category      
 12  h_edo               165199 non-null  category      
 13  Estado_cve_agencia  165199 non-nul

## Data Significance & Data Selection 

In [404]:
df = data_3.copy()

In [405]:
df.head()

,ID_Reserva,h_res_fec,h_num_adu,h_num_men,h_tot_hab,ID_Paquete,ID_Segmento_Comp,ID_Agencia,ID_Tipo_Habitacion,ID_canal,h_fec_lld,h_can_res,h_edo,Estado_cve_agencia,Pais_cod_agencia,tarif,days_diff_res_lld,h_num_per,Pais_Origen,Cancellation
0,0,2019-08-16,2,0,1,1,14,112,25,10,2019-11-07,09,EMX,EMX,PMX,2.693467e+05,83,2,1,0
1,1,2019-10-22,4,0,1,2,14,112,25,10,2019-12-01,09,EGT,EMX,PMX,2.693467e+05,40,4,1,0
2,2,2019-10-28,2,0,1,1,14,112,23,10,2019-12-01,09,EMC,EMX,PMX,1.988246e+05,34,2,1,0
3,3,2019-10-28,3,0,1,2,14,112,24,10,2019-12-08,09,EGT,EMX,PMX,1.742958e+05,41,3,1,0
4,4,2019-10-28,2,0,1,1,5,14,4,13,2019-12-08,12,EMX,EQR,PMX,3.608777e+06,41,2,1,0


### Análisis de variables categóricas y numéricas

En esta sección se realiza un análisis exploratorio de las variables categóricas y numéricas presentes en el conjunto de datos. Para las variables numéricas, se examinan sus distribuciones, valores atípicos y estadísticas descriptivas como la media, mediana, desviación estándar, valores mínimos y máximos. Esto permite identificar posibles anomalías, sesgos o transformaciones necesarias para el modelado.

En cuanto a las variables categóricas, se analiza la frecuencia de cada categoría, identificando aquellas con alta cardinalidad o categorías poco representadas. 

In [406]:
df_num = df[['h_num_adu', 'h_num_men', 'h_tot_hab', 'tarif', 'days_diff_res_lld', 'h_num_per']]

In [ ]:
corr_matrix = df_num.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matriz de correclacion de variables numericas')
plt.show()

In [408]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

scaled_values = scaler.fit_transform(df_num)

scaled_df = pd.DataFrame(scaled_values, columns=df_num.columns)

In [409]:
num_cols = data_3.select_dtypes(include=['int64', 'float64']).columns.drop('Cancellation')
cat_cols = data_3.select_dtypes(include=['category', 'object', 'bool']).columns

relevance_results = {}

In [410]:
for col in num_cols:
    group0 = data_3[data_3['Cancellation'] == 0][col]
    group1 = data_3[data_3['Cancellation'] == 1][col]

    # Prueba t-student
    stat_t, p_value_t = ttest_ind(group0, group1, nan_policy='omit')
    print(f"T-test for {col}:")
    print(f"Media grupo 0: {group0.mean():.2f}")
    print(f"Media grupo 1: {group1.mean():.2f}")
    print(f"t-statistic: {stat_t:.4f}, p-value: {p_value_t:.4e}")

    # Prueba U de Mann-Whitney (Qhiney U)
    stat_u, p_value_u = mannwhitneyu(group0, group1, alternative='two-sided')
    print(f"\nMann-Whitney U test for {col}:")
    print(f"U statistic: {stat_u:.4f}, p-value: {p_value_u:.4e}")

T-test for h_num_adu:
Media grupo 0: 1.08
Media grupo 1: 1.17
t-statistic: -11.2543, p-value: 2.2616e-29

Mann-Whitney U test for h_num_adu:
U statistic: 2205105747.5000, p-value: 3.4882e-05
T-test for h_num_men:
Media grupo 0: 0.03
Media grupo 1: 0.08
t-statistic: -33.5277, p-value: 1.2818e-245



Mann-Whitney U test for h_num_men:
U statistic: 2164517718.0000, p-value: 2.2320e-240
T-test for h_tot_hab:
Media grupo 0: 0.50
Media grupo 1: 0.52
t-statistic: -5.9173, p-value: 3.2783e-09

Mann-Whitney U test for h_tot_hab:
U statistic: 2216266589.0000, p-value: 7.1522e-03
T-test for days_diff_res_lld:
Media grupo 0: 55.42
Media grupo 1: 79.43
t-statistic: -60.7482, p-value: 0.0000e+00

Mann-Whitney U test for days_diff_res_lld:
U statistic: 1691355073.5000, p-value: 0.0000e+00
T-test for h_num_per:
Media grupo 0: 1.11
Media grupo 1: 1.25
t-statistic: -17.3839, p-value: 1.2562e-67

Mann-Whitney U test for h_num_per:
U statistic: 2187928065.0000, p-value: 6.6416e-11
T-test for Pais_Origen:
Media grupo 0: 0.99
Media grupo 1: 0.99
t-statistic: 5.5749, p-value: 2.4810e-08

Mann-Whitney U test for Pais_Origen:
U statistic: 2241656513.0000, p-value: 2.4844e-08


Algunas de las variables numéricas se conservaron porque los resultados de las pruebas estadísticas (t-student y Mann-Whitney) muestran diferencias significativas en la media o distribución de estas variables entre los grupos de cancelación y no cancelación (p-valores muy bajos o igual a cero), lo que sugiere que son relevantes para predecir la cancelacion de una reservacion. Por lo tanto, ninguna variable es redundante ni irrelevante. En el resto de estas, especificamente ID_Reserva', 'h_num_men', 'h_num_adu', 'h_num_per', la matriz de correlación muestra una alta correlación entre sí (descartando el análisis de la prueba t-student y Mann-Whitney), por lo que se decidió descartarlas para evitar redundancia y multicolinealidad.

In [411]:
for col in cat_cols:
    print(f"\nVariable: {col}")
    contingency = pd.crosstab(data_3[col], data_3['Cancellation'])
    chi2, p, dof, expected = stats.chi2_contingency(contingency)
    print(f"Chi2: {chi2:.4f}, p-value: {p:.4e}, dof: {dof}")


Variable: ID_Reserva
Chi2: 165199.0000, p-value: 4.9884e-01, dof: 165198

Variable: ID_Paquete
Chi2: 9589.7603, p-value: 0.0000e+00, dof: 4

Variable: ID_Segmento_Comp
Chi2: 6542.4829, p-value: 0.0000e+00, dof: 19

Variable: ID_Agencia
Chi2: 10671.5992, p-value: 0.0000e+00, dof: 109

Variable: ID_Tipo_Habitacion
Chi2: 17345.8180, p-value: 0.0000e+00, dof: 27

Variable: ID_canal
Chi2: 6033.0789, p-value: 0.0000e+00, dof: 13

Variable: h_can_res
Chi2: 6033.0789, p-value: 0.0000e+00, dof: 13

Variable: h_edo
Chi2: 65853.0059, p-value: 0.0000e+00, dof: 146

Variable: Estado_cve_agencia
Chi2: 3823.1788, p-value: 0.0000e+00, dof: 20

Variable: Pais_cod_agencia
Chi2: 561.1371, p-value: 2.6784e-121, dof: 3

Variable: tarif
Chi2: 17345.8180, p-value: 0.0000e+00, dof: 27


SE conservan las variables actegoricas ya que al realizar pruebas de significancia estadística (Chi-cuadrado de independencia), se observó que presentan asociaciones significativas con la variable objetivo (`Cancellation`). Esto indica que la distribución de las categorías difiere entre los grupos de cancelación y no cancelación, aportando información relevante para la predicción. 

En la celda se convierte la columna `h_fec_lld` a formato de fecha y se extraen tres nuevas variables.
mes (`mes_llegada`), año (`year_llegada`) y día (`dia_llegada`) de la llegada. 

Finalmente, se elimina la columna original `h_fec_lld` del DataFrame.

In [412]:
data_3['h_fec_lld'] = pd.to_datetime(data_3['h_fec_lld'], errors='coerce')

data_3['mes_llegada'] = data_3['h_fec_lld'].dt.month
data_3['year_llegada'] = data_3['h_fec_lld'].dt.year
data_3['dia_llegada'] = data_3['h_fec_lld'].dt.day

data_3.drop(columns=['h_fec_lld'],inplace=True)

In [413]:
from scipy.stats import chi2_contingency

chi_vars = ['mes_llegada', 'year_llegada', 'dia_llegada']

for var in chi_vars:
    contingency = pd.crosstab(data_3[var], data_3['Cancellation'])
    chi2, p, dof, expected = chi2_contingency(contingency)
    print(f"Variable: {var}")
    print(f"Chi2: {chi2:.4f}, p-value: {p:.4e}, dof: {dof}\n")

Variable: mes_llegada
Chi2: 3540.8325, p-value: 0.0000e+00, dof: 11

Variable: year_llegada
Chi2: 7305.3785, p-value: 0.0000e+00, dof: 2

Variable: dia_llegada
Chi2: 868.4943, p-value: 2.5761e-163, dof: 30



Se aplicó la prueba de Chi-cuadrado de independencia para evaluar la relación entre las variables temporales extraídas de la fecha de llegada (`mes_llegada`, `year_llegada`, `dia_llegada`) y la variable objetivo de cancelación (`Cancellation`). Los resultados muestran que las tres variables cuentan con un valor p muy bajo, lo que indica que hay una asociación significativa entre ellas y la probabilidad de cancelación.

In [414]:
data_3 = data_3.drop(['ID_Reserva', 'h_num_per'], axis=1)
data_3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 165199 entries, 0 to 203001
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   h_res_fec           165199 non-null  datetime64[ns]
 1   h_num_adu           165199 non-null  int64         
 2   h_num_men           165199 non-null  int64         
 3   h_tot_hab           165199 non-null  int64         
 4   ID_Paquete          165199 non-null  category      
 5   ID_Segmento_Comp    165199 non-null  category      
 6   ID_Agencia          165199 non-null  category      
 7   ID_Tipo_Habitacion  165199 non-null  category      
 8   ID_canal            165199 non-null  category      
 9   h_can_res           165199 non-null  category      
 10  h_edo               165199 non-null  category      
 11  Estado_cve_agencia  165199 non-null  category      
 12  Pais_cod_agencia    165199 non-null  category      
 13  tarif               165199 non-nul

### Segmentar datos en prueba y entrenamiento.

In [415]:
# Separar X y y para el entrenamiento del modelo
y = data_3['Cancellation'].values # target
X_raw = data_3.drop('Cancellation', axis=1) # features

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

In [416]:
X_train.shape, y_train.shape

((132159, 19), (132159,))

In [417]:
data_3['ID_Agencia'].value_counts()

ID_Agencia
112    24327
99     21250
86     12564
14     11862
116     8836
       ...  
75         0
108        0
13         0
63         0
117        0
Name: count, Length: 120, dtype: int64

In [418]:
data_3.columns

Index(['h_res_fec', 'h_num_adu', 'h_num_men', 'h_tot_hab', 'ID_Paquete',
       'ID_Segmento_Comp', 'ID_Agencia', 'ID_Tipo_Habitacion', 'ID_canal',
       'h_can_res', 'h_edo', 'Estado_cve_agencia', 'Pais_cod_agencia', 'tarif',
       'days_diff_res_lld', 'Pais_Origen', 'Cancellation', 'mes_llegada',
       'year_llegada', 'dia_llegada'],
      dtype='object')

### Ajustrar codificadores y escaladores con los datos de entrenamiento.

In [ ]:
# one hot encoding a variables categoricas nominales de cardinalidad moderada
ONE_vars = ['ID_Paquete', 'ID_Segmento_Comp', 'ID_Agencia', 'ID_Tipo_Habitacion', 'ID_canal', 'h_can_res', 'Estado_cve_agencia', 'Pais_cod_agencia', 'Pais_Origen']

encoder_ONE = OneHotEncoder(drop='first', handle_unknown='ignore', sparse=False)
encoder_ONE.fit(X_train[ONE_vars])

encoded_ONE_data = encoder_ONE.transform(X_train[ONE_vars])
encoded_ONE_df = pd.DataFrame(encoded_ONE_data, columns=encoder_ONE.get_feature_names_out())

encoded_ONE_data_2 = encoder_ONE.transform(X_test[ONE_vars])
encoded_ONE_df_2 = pd.DataFrame(encoded_ONE_data_2, columns=encoder_ONE.get_feature_names_out())


X_train = pd.concat([X_train.drop(columns=ONE_vars).reset_index(drop=True), encoded_ONE_df.reset_index(drop=True)], axis=1)
X_test = pd.concat([X_test.drop(columns=ONE_vars).reset_index(drop=True), encoded_ONE_df_2.reset_index(drop=True)], axis=1)

In [273]:
import joblib

# Save the encoder
joblib.dump(encoder_ONE, "encoder_ONE.pkl")

['encoder_ONE.pkl']

Las variables seleccionadas para aplicar one hot encoding (`ID_Paquete`, `ID_Segmento_Comp`, `ID_Agencia`, `ID_Tipo_Habitacion`, `ID_canal`, `h_can_res`, `Estado_cve_agencia`, `Pais_cod_agencia`, `Pais_Origen`) con el fin de transformar estas variables en un formato adecuado para los modelos de machine learning. 

In [420]:
# Target encoding a variable 'h_edo'
te = TargetEncoder(cols=['h_edo'], handle_unknown=0)
te.fit(X_train['h_edo'], y_train)


X_train['h_edo_target_enc'] = te.transform(X_train['h_edo'])
X_train['h_edo_target_enc'] = X_train['h_edo_target_enc'].fillna(0)
X_train = X_train.drop('h_edo', axis=1)

X_test['h_edo_target_enc'] = te.transform(X_test['h_edo'])
X_test['h_edo_target_enc'] = X_test['h_edo_target_enc'].fillna(0)
X_test = X_test.drop('h_edo', axis=1)

In [421]:
# Save the encoder
joblib.dump(te, "target_encoder.pkl")

['target_encoder.pkl']

La variable `h_edo` se codificó mediante target encoding porque es una variable categórica con alta cardinalidad (muchos valores únicos). 

In [422]:
cols_to_normalize = ['tarif', 'days_diff_res_lld', 'h_tot_hab', 'mes_llegada', 'year_llegada', 'dia_llegada', 'h_num_adu', 'h_num_men']

scaler = StandardScaler()
scaler.fit(X_train[cols_to_normalize])


X_train[cols_to_normalize] = scaler.transform(X_train[cols_to_normalize])
X_train[cols_to_normalize].head()

X_test[cols_to_normalize] = scaler.transform(X_test[cols_to_normalize])
X_test[cols_to_normalize].head()


,tarif,days_diff_res_lld,h_tot_hab,mes_llegada,year_llegada,dia_llegada,h_num_adu,h_num_men
0,1.208879,-0.659159,-0.990999,0.245679,-0.549226,-0.443177,-0.898841,-0.139751
1,1.208879,-0.689570,0.976454,0.245679,-0.549226,1.295864,0.735403,-0.139751
2,-0.856074,-0.826421,0.976454,1.439832,-0.549226,0.252439,0.735403,-0.139751
3,1.208879,-0.674364,-0.990999,-0.649935,-0.549226,-1.254729,-0.898841,-0.139751
4,0.543407,1.180731,-0.990999,0.245679,1.820436,0.484311,-0.898841,-0.139751


In [423]:
# Save the encoder
joblib.dump(scaler, "standard_scaler.pkl")

['standard_scaler.pkl']

In [424]:
X_train.drop(columns=['h_res_fec'], inplace=True)
X_test.drop(columns=['h_res_fec'], inplace=True)

In [426]:
X_train.to_parquet('data_TDA_X_Train.parquet')
pd.DataFrame(y_train).to_parquet('data_TDA_y_Train.parquet')

X_test.to_parquet('data_TDA_X_test.parquet')
pd.DataFrame(y_test).to_parquet('data_TDA_y_test.parquet')